# PAPIT Standalone Colab Notebook

This notebook is fully self-contained and does not depend on any local repository files.

## 1) Environment Setup

Run this first. In Colab, switch runtime to GPU for best speed.

This setup also installs plotting and optional OCR dependencies for later cells.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

# Install dependencies needed by this notebook only.
!pip -q install -U pip
!pip -q install torch transformers pillow matplotlib opencv-python-headless easyocr
print('Dependencies installed.')

## 2) Define PAPIT Pipeline (Standalone + Position Remap + LLM Projection)

This cell defines a self-contained PAPIT implementation with:
- prompt-aware top-k pruning
- optional anchor token
- position remapping for selected patches
- a lightweight projection utility for LLaVA-style input preparation

In [ ]:
from dataclasses import dataclass
from typing import Literal, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

AnchorStrategy = Literal['none', 'global_mean', 'dropped_mean']

@dataclass(slots=True)
class PAPITConfig:
    model_id: str = 'openai/clip-vit-base-patch32'
    retention_ratio: float = 0.5
    anchor_strategy: AnchorStrategy = 'global_mean'
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

@dataclass(slots=True)
class PAPITOutput:
    patch_tokens: torch.Tensor
    text_embedding: torch.Tensor
    scores: torch.Tensor
    topk_indices: torch.Tensor
    topk_scores: torch.Tensor
    pruned_tokens: torch.Tensor
    coords: list[tuple[int, int]]
    new_position_ids: torch.Tensor
    selected_position_ids: torch.Tensor

class PromptAwarePruner:
    def __init__(self, config: PAPITConfig) -> None:
        self.config = config
        self.processor = CLIPProcessor.from_pretrained(config.model_id)
        self.model = CLIPModel.from_pretrained(config.model_id).to(config.device)
        self.model.eval()

    @torch.no_grad()
    def run(self, image_path: str, prompt: str) -> PAPITOutput:
        image = Image.open(image_path).convert('RGB')
        inputs = self.processor(text=[prompt], images=image, return_tensors='pt')
        inputs = {k: v.to(self.config.device) for k, v in inputs.items()}

        vision_outputs = self.model.vision_model(
            pixel_values=inputs['pixel_values'],
            output_hidden_states=False,
            return_dict=True,
        )
        # Drop CLS token: keep only patch tokens in vision hidden space.
        raw_patch_tokens = vision_outputs.last_hidden_state[:, 1:, :].squeeze(0)
        # Map patch tokens to CLIP shared embedding space for cross-modal scoring.
        patch_tokens = self.model.visual_projection(raw_patch_tokens)

        text_embedding = self._extract_text_embedding(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
        )

        scores = self._compute_scores(patch_tokens, text_embedding)
        topk_scores, topk_indices = self._select_topk(scores)
        selected_tokens = patch_tokens[topk_indices]
        pruned_tokens = self._append_anchor(selected_tokens, patch_tokens, topk_indices)
        coords = self._indices_to_coords(topk_indices.cpu())

        selected_position_ids, new_position_ids = self._remap_positions(topk_indices)

        return PAPITOutput(
            patch_tokens=patch_tokens,
            text_embedding=text_embedding,
            scores=scores,
            topk_indices=topk_indices,
            topk_scores=topk_scores,
            pruned_tokens=pruned_tokens,
            coords=coords,
            new_position_ids=new_position_ids,
            selected_position_ids=selected_position_ids,
        )

    def _extract_text_embedding(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        # Compatible across transformers versions where get_text_features behavior can vary.
        text_features = self.model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        if isinstance(text_features, torch.Tensor):
            return text_features.squeeze(0)

        text_outputs = self.model.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        pooled = text_outputs.pooler_output
        projected = self.model.text_projection(pooled)
        return projected.squeeze(0)

    def _compute_scores(self, patch_tokens: torch.Tensor, text_embedding: torch.Tensor) -> torch.Tensor:
        if patch_tokens.shape[-1] != text_embedding.shape[-1]:
            raise ValueError(
                f'Dimension mismatch before scoring: patches={patch_tokens.shape[-1]}, text={text_embedding.shape[-1]}'
            )
        patch_norm = F.normalize(patch_tokens, p=2, dim=-1)
        text_norm = F.normalize(text_embedding, p=2, dim=-1)
        return patch_norm @ text_norm

    def _select_topk(self, scores: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        ratio = max(0.0, min(1.0, self.config.retention_ratio))
        token_count = scores.shape[0]
        k = max(1, int(round(token_count * ratio)))
        topk_scores, topk_indices = torch.topk(scores, k=k, largest=True, sorted=True)
        return topk_scores, topk_indices

    def _append_anchor(
        self,
        selected_tokens: torch.Tensor,
        patch_tokens: torch.Tensor,
        selected_indices: torch.Tensor,
    ) -> torch.Tensor:
        if self.config.anchor_strategy == 'none':
            return selected_tokens

        if self.config.anchor_strategy == 'global_mean':
            anchor = patch_tokens.mean(dim=0, keepdim=True)
            return torch.cat([selected_tokens, anchor], dim=0)

        if self.config.anchor_strategy == 'dropped_mean':
            mask = torch.ones(patch_tokens.shape[0], device=patch_tokens.device, dtype=torch.bool)
            mask[selected_indices] = False
            anchor = patch_tokens[mask].mean(dim=0, keepdim=True) if mask.any() else patch_tokens.mean(dim=0, keepdim=True)
            return torch.cat([selected_tokens, anchor], dim=0)

        raise ValueError(f'Unknown anchor strategy: {self.config.anchor_strategy}')

    def _indices_to_coords(self, indices: torch.Tensor) -> list[tuple[int, int]]:
        vision_cfg = self.model.vision_model.config
        image_size = getattr(vision_cfg, 'image_size', 224)
        patch_size = getattr(vision_cfg, 'patch_size', 16)
        grid_size = max(int(image_size // patch_size), 1)

        coords = []
        for idx in indices.tolist():
            row = idx // grid_size
            col = idx % grid_size
            coords.append((row, col))
        return coords

    def _remap_positions(self, selected_indices: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # selected_position_ids: original patch positions
        # new_position_ids: compact 0..k-1 index after pruning
        selected_position_ids = selected_indices.clone()
        new_position_ids = torch.arange(
            selected_indices.shape[0],
            device=selected_indices.device,
            dtype=torch.long,
        )
        return selected_position_ids, new_position_ids

    def project_for_llm(
        self,
        pruned_tokens: torch.Tensor,
        llm_hidden_dim: int = 4096,
        projector: Optional[nn.Module] = None,
    ) -> tuple[torch.Tensor, nn.Module]:
        # Utility to prepare pruned image tokens for LLM input space.
        token_dim = pruned_tokens.shape[-1]
        if projector is None:
            projector = nn.Linear(token_dim, llm_hidden_dim).to(pruned_tokens.device)
        projected = projector(pruned_tokens)
        return projected, projector

print('Standalone PAPIT classes loaded (with position remap + LLM projection helper).')

## 3) Check GPU Availability

In [ ]:
import torch
print('torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

## 4) Upload an Image

In [ ]:
from pathlib import Path
image_path = Path('sample.jpg')

if not image_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            image_path = Path(next(iter(uploaded.keys())))
    except Exception:
        pass

if not image_path.exists():
    raise FileNotFoundError('No image found. Upload one file in this cell.')

print('Using image:', image_path.resolve())

## 5) Run PAPIT Once (with Position Remap + LLM Projection Shape)

In [ ]:
import json

prompt = 'Read the text on the sign.'
retention = 0.5
anchor = 'global_mean'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = PAPITConfig(
    model_id='openai/clip-vit-base-patch32',
    retention_ratio=retention,
    anchor_strategy=anchor,
    device=device,
)
pruner = PromptAwarePruner(config)
result = pruner.run(str(image_path), prompt)

# Build LLaVA-style projected token embeddings (shape check).
image_embeds_for_llm, projector = pruner.project_for_llm(
    result.pruned_tokens,
    llm_hidden_dim=4096,
    projector=None,
)

summary = {
    'image': str(image_path),
    'prompt': prompt,
    'device': device,
    'retention_ratio': retention,
    'anchor_strategy': anchor,
    'total_patch_tokens': int(result.patch_tokens.shape[0]),
    'selected_patch_tokens': int(result.topk_indices.shape[0]),
    'final_sequence_length': int(result.pruned_tokens.shape[0]),
    'projected_llm_shape': list(image_embeds_for_llm.shape),
    'topk_indices': [int(x) for x in result.topk_indices.detach().cpu().tolist()[:20]],
    'topk_scores': [float(x) for x in result.topk_scores.detach().cpu().tolist()[:20]],
    'topk_coords': [[int(r), int(c)] for r, c in result.coords[:20]],
    'original_position_ids': [int(x) for x in result.selected_position_ids.detach().cpu().tolist()[:20]],
    'new_position_ids': [int(x) for x in result.new_position_ids.detach().cpu().tolist()[:20]],
}

print(json.dumps(summary, indent=2))

## 6) Optional: CPU vs GPU Timing (Pruning Stage)

In [ ]:
import time

def timed_run(device: str, warmup: int = 1, runs: int = 3):
    cfg = PAPITConfig(
        model_id='openai/clip-vit-base-patch32',
        retention_ratio=0.5,
        anchor_strategy='global_mean',
        device=device,
    )
    pr = PromptAwarePruner(cfg)

    for _ in range(warmup):
        _ = pr.run(str(image_path), prompt)

    start = time.perf_counter()
    for _ in range(runs):
        _ = pr.run(str(image_path), prompt)
    elapsed = time.perf_counter() - start
    return elapsed / runs

cpu_time = timed_run('cpu')
print(f'CPU avg time: {cpu_time:.4f}s')

if torch.cuda.is_available():
    gpu_time = timed_run('cuda')
    print(f'GPU avg time: {gpu_time:.4f}s')
    print(f'Speedup: {cpu_time / gpu_time:.2f}x')
else:
    print('CUDA not available. Skipping GPU timing.')

## 7) Save JSON Summary

In [ ]:
import json
from pathlib import Path

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'colab_run_summary.json'

with out_path.open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Saved:', out_path.resolve())

try:
    from google.colab import files
    files.download(str(out_path))
except Exception:
    pass

## 8) Visualize Selected Patches

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

img = Image.open(image_path).convert('RGB')
w, h = img.size

vision_cfg = pruner.model.vision_model.config
clip_image_size = int(getattr(vision_cfg, 'image_size', 224))
patch_size = int(getattr(vision_cfg, 'patch_size', 16))
grid_size = max(clip_image_size // patch_size, 1)

cell_w = w / grid_size
cell_h = h / grid_size

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img)
ax.set_title('Selected patches (red)')

for r, c in result.coords:
    rect = patches.Rectangle(
        (c * cell_w, r * cell_h),
        cell_w,
        cell_h,
        linewidth=1.5,
        edgecolor='red',
        facecolor='none',
    )
    ax.add_patch(rect)

ax.axis('off')
plt.show()

## 9) Optional OCR-Guided Retention (TextVQA-style)

This cell finds text boxes with EasyOCR, maps them to patch indices, and forces those patches to be retained while keeping the same pruning budget.

In [ ]:
import numpy as np
import easyocr

def ocr_forced_indices(image_path, grid_size, lang_list=None):
    if lang_list is None:
        lang_list = ['en']
    reader = easyocr.Reader(lang_list, gpu=torch.cuda.is_available())
    img_np = np.array(Image.open(image_path).convert('RGB'))
    results = reader.readtext(img_np)

    h, w = img_np.shape[:2]
    cell_w = w / grid_size
    cell_h = h / grid_size

    forced = set()
    for item in results:
        box = item[0]
        xs = [p[0] for p in box]
        ys = [p[1] for p in box]
        x0, x1 = max(min(xs), 0), min(max(xs), w - 1)
        y0, y1 = max(min(ys), 0), min(max(ys), h - 1)

        c0 = int(x0 // cell_w)
        c1 = int(x1 // cell_w)
        r0 = int(y0 // cell_h)
        r1 = int(y1 // cell_h)

        c0 = max(0, min(c0, grid_size - 1))
        c1 = max(0, min(c1, grid_size - 1))
        r0 = max(0, min(r0, grid_size - 1))
        r1 = max(0, min(r1, grid_size - 1))

        for r in range(r0, r1 + 1):
            for c in range(c0, c1 + 1):
                forced.add(r * grid_size + c)
    return sorted(forced), results

def merge_topk_with_forced(scores, topk_indices, k, forced_indices):
    forced_set = set(int(i) for i in forced_indices)
    ranked = torch.argsort(scores, descending=True).detach().cpu().tolist()

    final = []
    for idx in topk_indices.detach().cpu().tolist():
        if idx in forced_set and idx not in final:
            final.append(idx)

    for idx in ranked:
        if idx in forced_set and idx not in final:
            final.append(idx)

    for idx in ranked:
        if idx not in final:
            final.append(idx)
        if len(final) >= k:
            break

    return final[:k]

vision_cfg = pruner.model.vision_model.config
clip_image_size = int(getattr(vision_cfg, 'image_size', 224))
patch_size = int(getattr(vision_cfg, 'patch_size', 16))
grid_size = max(clip_image_size // patch_size, 1)

forced_indices, ocr_results = ocr_forced_indices(str(image_path), grid_size=grid_size, lang_list=['en'])
k = int(result.topk_indices.shape[0])
ocr_topk = merge_topk_with_forced(result.scores, result.topk_indices, k=k, forced_indices=forced_indices)

ocr_summary = {
    'ocr_detected_regions': len(ocr_results),
    'forced_patch_count': len(forced_indices),
    'base_k': k,
    'ocr_guided_topk_count': len(ocr_topk),
    'ocr_guided_topk_first20': ocr_topk[:20],
}
print(json.dumps(ocr_summary, indent=2))

In [ ]:
# Visual compare: base selected patches (red) vs OCR-guided patches (cyan)
img = Image.open(image_path).convert('RGB')
w, h = img.size
cell_w = w / grid_size
cell_h = h / grid_size

base_set = set(int(x) for x in result.topk_indices.detach().cpu().tolist())
ocr_set = set(int(x) for x in ocr_topk)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(img)
ax.set_title('Base top-k (red) vs OCR-guided top-k (cyan)')

for idx in sorted(base_set):
    r, c = divmod(idx, grid_size)
    rect = patches.Rectangle(
        (c * cell_w, r * cell_h),
        cell_w,
        cell_h,
        linewidth=1.2,
        edgecolor='red',
        facecolor='none',
    )
    ax.add_patch(rect)

for idx in sorted(ocr_set):
    r, c = divmod(idx, grid_size)
    rect = patches.Rectangle(
        (c * cell_w, r * cell_h),
        cell_w,
        cell_h,
        linewidth=1.0,
        edgecolor='cyan',
        facecolor='none',
    )
    ax.add_patch(rect)

ax.axis('off')
plt.show()

## 10) Quantify Overlap and OCR Coverage

Use this to interpret red vs cyan boxes numerically.

In [ ]:
base_set = set(int(x) for x in result.topk_indices.detach().cpu().tolist())
ocr_set = set(int(x) for x in ocr_topk)
forced_set = set(int(x) for x in forced_indices)

intersection = base_set & ocr_set
union = base_set | ocr_set

def pct(num, den):
    return 0.0 if den == 0 else 100.0 * num / den

metrics = {
    'k': len(base_set),
    'forced_patch_count': len(forced_set),
    'base_vs_ocr_intersection': len(intersection),
    'base_vs_ocr_union': len(union),
    'jaccard_base_vs_ocr': 0.0 if len(union) == 0 else len(intersection) / len(union),
    'forced_covered_by_base_count': len(base_set & forced_set),
    'forced_covered_by_ocr_count': len(ocr_set & forced_set),
    'forced_covered_by_base_pct': pct(len(base_set & forced_set), len(forced_set)),
    'forced_covered_by_ocr_pct': pct(len(ocr_set & forced_set), len(forced_set)),
    'coverage_gain_pct_points': pct(len(ocr_set & forced_set), len(forced_set)) - pct(len(base_set & forced_set), len(forced_set)),
}

print(json.dumps(metrics, indent=2))

## 11) Retention Sweep for OCR Coverage

Runs multiple retention ratios and reports how much OCR text-region coverage improves with OCR-guided selection.

In [ ]:
import pandas as pd

ratios = [0.25, 0.5, 0.75]
rows = []

for r in ratios:
    cfg = PAPITConfig(
        model_id='openai/clip-vit-base-patch32',
        retention_ratio=r,
        anchor_strategy='global_mean',
        device=device,
    )
    pr = PromptAwarePruner(cfg)
    out = pr.run(str(image_path), prompt)

    base = set(int(x) for x in out.topk_indices.detach().cpu().tolist())
    k = len(base)
    ocr_topk_local = merge_topk_with_forced(out.scores, out.topk_indices, k=k, forced_indices=forced_indices)
    ocr_local = set(int(x) for x in ocr_topk_local)
    forced_local = set(int(x) for x in forced_indices)

    base_cov = pct(len(base & forced_local), len(forced_local))
    ocr_cov = pct(len(ocr_local & forced_local), len(forced_local))

    rows.append({
        'retention_ratio': r,
        'k': k,
        'forced_patch_count': len(forced_local),
        'base_forced_coverage_pct': base_cov,
        'ocr_guided_forced_coverage_pct': ocr_cov,
        'coverage_gain_pct_points': ocr_cov - base_cov,
    })

df = pd.DataFrame(rows)
display(df)

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / 'ocr_coverage_sweep.csv'
df.to_csv(csv_path, index=False)
print('Saved sweep table:', csv_path.resolve())

## 12) Optional End-Task QA Check (Original vs Pruned vs OCR-Guided)

This section runs a lightweight VQA model to compare answer quality after pruning.

Note: this is a practical proxy for end-task quality, not full LLaVA integration yet.

In [ ]:
from transformers import BlipProcessor, BlipForQuestionAnswering
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def build_pruned_image(image_path, kept_indices, grid_size):
    img = Image.open(image_path).convert('RGB')
    arr = np.array(img).copy()
    h, w = arr.shape[:2]
    cell_w = w / grid_size
    cell_h = h / grid_size

    kept = set(int(x) for x in kept_indices)
    for idx in range(grid_size * grid_size):
        if idx in kept:
            continue
        r, c = divmod(idx, grid_size)
        x0 = int(c * cell_w)
        x1 = int((c + 1) * cell_w)
        y0 = int(r * cell_h)
        y1 = int((r + 1) * cell_h)
        arr[y0:y1, x0:x1] = 0
    return Image.fromarray(arr)

qa_model_id = 'Salesforce/blip-vqa-base'
qa_device = 'cuda' if torch.cuda.is_available() else 'cpu'

qa_processor = BlipProcessor.from_pretrained(qa_model_id)
qa_model = BlipForQuestionAnswering.from_pretrained(qa_model_id).to(qa_device)
qa_model.eval()

print('Loaded QA model:', qa_model_id, 'on', qa_device)

In [ ]:
def answer_with_blip(image_pil, question, max_new_tokens=20):
    inputs = qa_processor(images=image_pil, text=question, return_tensors='pt')
    inputs = {k: v.to(qa_device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = qa_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return qa_processor.decode(output_ids[0], skip_special_tokens=True).strip()

orig_img = Image.open(image_path).convert('RGB')
base_pruned_img = build_pruned_image(str(image_path), result.topk_indices.detach().cpu().tolist(), grid_size)
ocr_pruned_img = build_pruned_image(str(image_path), ocr_topk, grid_size)

question = prompt
ans_orig = answer_with_blip(orig_img, question)
ans_base = answer_with_blip(base_pruned_img, question)
ans_ocr = answer_with_blip(ocr_pruned_img, question)

qa_compare = {
    'question': question,
    'answer_original': ans_orig,
    'answer_base_pruned': ans_base,
    'answer_ocr_guided_pruned': ans_ocr,
}
print(json.dumps(qa_compare, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(orig_img)
axes[0].set_title(f'Original\nAns: {ans_orig}')
axes[0].axis('off')

axes[1].imshow(base_pruned_img)
axes[1].set_title(f'Base Pruned\nAns: {ans_base}')
axes[1].axis('off')

axes[2].imshow(ocr_pruned_img)
axes[2].set_title(f'OCR-guided Pruned\nAns: {ans_ocr}')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 13) Batch Benchmark Runner (Base vs OCR-Guided vs Random)

This section automates evaluation across many image-question samples and exports CSV reports.

Expected CSV columns:
- image_path
- question
- answer (optional ground truth)

In [ ]:
import os
import time
import random
import pandas as pd
import numpy as np
from pathlib import Path

def normalize_text(s):
    if s is None:
        return ''
    s = str(s).strip().lower()
    for ch in [',', '.', '!', '?', ':', ';', '"', "'", '(', ')']:
        s = s.replace(ch, ' ')
    s = ' '.join(s.split())
    return s

def token_f1(pred, gold):
    p = normalize_text(pred).split()
    g = normalize_text(gold).split()
    if len(p) == 0 and len(g) == 0:
        return 1.0
    if len(p) == 0 or len(g) == 0:
        return 0.0
    p_count = {}
    g_count = {}
    for t in p:
        p_count[t] = p_count.get(t, 0) + 1
    for t in g:
        g_count[t] = g_count.get(t, 0) + 1
    overlap = 0
    for t, c in p_count.items():
        if t in g_count:
            overlap += min(c, g_count[t])
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(g)
    return 2 * precision * recall / (precision + recall)

def exact_match(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def random_topk_indices(total_tokens, k, seed=0):
    rng = random.Random(seed)
    idx = list(range(total_tokens))
    rng.shuffle(idx)
    idx = idx[:k]
    idx.sort()
    return idx

print('Batch benchmark helpers loaded.')

In [ ]:
# Create a small template CSV if you do not have one yet.
template_path = Path('benchmark_samples.csv')
if not template_path.exists():
    pd.DataFrame([
        {'image_path': str(image_path), 'question': prompt, 'answer': ''},
    ]).to_csv(template_path, index=False)
    print('Created template:', template_path.resolve())
else:
    print('Template exists:', template_path.resolve())

print('Edit benchmark_samples.csv with your own rows before running the next cell.')

In [ ]:
# Run batch benchmark.
dataset_csv = 'benchmark_samples.csv'
retention_list = [0.25, 0.5, 0.75]
anchor_strategy = 'global_mean'
max_samples = 20  # set None to run all
seed = 42

samples = pd.read_csv(dataset_csv)
required_cols = {'image_path', 'question'}
if not required_cols.issubset(set(samples.columns)):
    raise ValueError(f'Missing required columns in {dataset_csv}. Need: {required_cols}')

if max_samples is not None:
    samples = samples.head(max_samples).copy()

rows = []
t0_all = time.perf_counter()

for r in retention_list:
    cfg = PAPITConfig(
        model_id='openai/clip-vit-base-patch32',
        retention_ratio=r,
        anchor_strategy=anchor_strategy,
        device=device,
    )
    pr = PromptAwarePruner(cfg)

    for i, row in samples.iterrows():
        img_path = str(row['image_path'])
        q = str(row['question'])
        gold = row['answer'] if 'answer' in samples.columns else ''

        t0 = time.perf_counter()
        out = pr.run(img_path, q)
        prune_time = time.perf_counter() - t0

        total_tokens = int(out.patch_tokens.shape[0])
        k = int(out.topk_indices.shape[0])

        vision_cfg_local = pr.model.vision_model.config
        clip_image_size_local = int(getattr(vision_cfg_local, 'image_size', 224))
        patch_size_local = int(getattr(vision_cfg_local, 'patch_size', 16))
        grid_size_local = max(clip_image_size_local // patch_size_local, 1)

        forced_local, _ = ocr_forced_indices(img_path, grid_size_local, lang_list=['en'])
        forced_set = set(int(x) for x in forced_local)

        base_idx = [int(x) for x in out.topk_indices.detach().cpu().tolist()]
        ocr_idx = merge_topk_with_forced(out.scores, out.topk_indices, k=k, forced_indices=forced_local)
        rnd_idx = random_topk_indices(total_tokens, k=k, seed=seed + i + int(r * 1000))

        base_img = build_pruned_image(img_path, base_idx, grid_size_local)
        ocr_img = build_pruned_image(img_path, ocr_idx, grid_size_local)
        rnd_img = build_pruned_image(img_path, rnd_idx, grid_size_local)
        orig_img_local = Image.open(img_path).convert('RGB')

        ans_orig = answer_with_blip(orig_img_local, q)
        ans_base = answer_with_blip(base_img, q)
        ans_ocr = answer_with_blip(ocr_img, q)
        ans_rnd = answer_with_blip(rnd_img, q)

        base_cov = pct(len(set(base_idx) & forced_set), len(forced_set))
        ocr_cov = pct(len(set(ocr_idx) & forced_set), len(forced_set))
        rnd_cov = pct(len(set(rnd_idx) & forced_set), len(forced_set))

        row_out = {
            'sample_index': int(i),
            'image_path': img_path,
            'question': q,
            'retention_ratio': r,
            'k': k,
            'total_tokens': total_tokens,
            'forced_patch_count': len(forced_set),
            'prune_time_sec': prune_time,
            'answer_orig': ans_orig,
            'answer_base': ans_base,
            'answer_ocr': ans_ocr,
            'answer_random': ans_rnd,
            'base_cov_pct': base_cov,
            'ocr_cov_pct': ocr_cov,
            'random_cov_pct': rnd_cov,
            'base_same_as_orig': int(normalize_text(ans_base) == normalize_text(ans_orig)),
            'ocr_same_as_orig': int(normalize_text(ans_ocr) == normalize_text(ans_orig)),
            'random_same_as_orig': int(normalize_text(ans_rnd) == normalize_text(ans_orig)),
        }

        if isinstance(gold, str) and len(gold.strip()) > 0:
            row_out['gold_answer'] = gold
            row_out['orig_em'] = exact_match(ans_orig, gold)
            row_out['base_em'] = exact_match(ans_base, gold)
            row_out['ocr_em'] = exact_match(ans_ocr, gold)
            row_out['random_em'] = exact_match(ans_rnd, gold)
            row_out['orig_f1'] = token_f1(ans_orig, gold)
            row_out['base_f1'] = token_f1(ans_base, gold)
            row_out['ocr_f1'] = token_f1(ans_ocr, gold)
            row_out['random_f1'] = token_f1(ans_rnd, gold)

        rows.append(row_out)

elapsed_all = time.perf_counter() - t0_all
print(f'Batch run complete in {elapsed_all:.2f}s for {len(rows)} rows.')

results_df = pd.DataFrame(rows)
display(results_df.head(10))

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
detailed_csv = out_dir / 'benchmark_detailed.csv'
results_df.to_csv(detailed_csv, index=False)
print('Saved detailed results:', detailed_csv.resolve())

In [ ]:
# Aggregate summary and plot accuracy-efficiency curves.
summary_rows = []

for r, g in results_df.groupby('retention_ratio'):
    summary_rows.append({
        'retention_ratio': r,
        'n_samples': len(g),
        'avg_prune_time_sec': g['prune_time_sec'].mean(),
        'avg_base_cov_pct': g['base_cov_pct'].mean(),
        'avg_ocr_cov_pct': g['ocr_cov_pct'].mean(),
        'avg_random_cov_pct': g['random_cov_pct'].mean(),
        'avg_base_same_as_orig': g['base_same_as_orig'].mean(),
        'avg_ocr_same_as_orig': g['ocr_same_as_orig'].mean(),
        'avg_random_same_as_orig': g['random_same_as_orig'].mean(),
    })

    if 'base_em' in g.columns:
        summary_rows[-1]['avg_base_em'] = g['base_em'].mean()
        summary_rows[-1]['avg_ocr_em'] = g['ocr_em'].mean()
        summary_rows[-1]['avg_random_em'] = g['random_em'].mean()
        summary_rows[-1]['avg_base_f1'] = g['base_f1'].mean()
        summary_rows[-1]['avg_ocr_f1'] = g['ocr_f1'].mean()
        summary_rows[-1]['avg_random_f1'] = g['random_f1'].mean()

summary_df = pd.DataFrame(summary_rows).sort_values('retention_ratio')
display(summary_df)

summary_csv = out_dir / 'benchmark_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print('Saved summary results:', summary_csv.resolve())

# Curve 1: OCR/text coverage vs retention.
plt.figure(figsize=(7, 5))
plt.plot(summary_df['retention_ratio'], summary_df['avg_base_cov_pct'], marker='o', label='Base')
plt.plot(summary_df['retention_ratio'], summary_df['avg_ocr_cov_pct'], marker='o', label='OCR-guided')
plt.plot(summary_df['retention_ratio'], summary_df['avg_random_cov_pct'], marker='o', label='Random')
plt.xlabel('Retention Ratio')
plt.ylabel('Avg OCR Forced Coverage (%)')
plt.title('Coverage-Retention Curve')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# Curve 2: Answer consistency vs retention.
plt.figure(figsize=(7, 5))
plt.plot(summary_df['retention_ratio'], summary_df['avg_base_same_as_orig'], marker='o', label='Base')
plt.plot(summary_df['retention_ratio'], summary_df['avg_ocr_same_as_orig'], marker='o', label='OCR-guided')
plt.plot(summary_df['retention_ratio'], summary_df['avg_random_same_as_orig'], marker='o', label='Random')
plt.xlabel('Retention Ratio')
plt.ylabel('Avg Consistency to Original Answer')
plt.title('Answer-Consistency Curve')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 14) Optional LLaVA Sanity Check (Image-Level Proxy)

This optional section compares answers from a LLaVA model on original vs pruned images.

Note: this is image-level proxy evaluation, not token-level LLaVA surgery.

In [ ]:
# Optional heavy model cell: run only if you have enough VRAM.
# Suggested models:
# - llava-hf/llava-1.5-7b-hf (heavy)

# Example skeleton (uncomment to run):
'''
from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch

llava_model_id = 'llava-hf/llava-1.5-7b-hf'
llava_processor = AutoProcessor.from_pretrained(llava_model_id)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    llava_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
).eval()

def answer_with_llava(image_pil, question):
    prompt_text = f'USER: <image>\n{question}\nASSISTANT:'
    inputs = llava_processor(text=prompt_text, images=image_pil, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = llava_model.generate(**inputs, max_new_tokens=64)
    return llava_processor.batch_decode(out_ids, skip_special_tokens=True)[0]

llava_ans_orig = answer_with_llava(orig_img, question)
llava_ans_base = answer_with_llava(base_pruned_img, question)
llava_ans_ocr = answer_with_llava(ocr_pruned_img, question)

print('LLaVA original:', llava_ans_orig)
print('LLaVA base-pruned:', llava_ans_base)
print('LLaVA ocr-guided:', llava_ans_ocr)
'''

## 15) Risk-Awareness Prototype (Safety Keep + Instruction Neutralization)

Implements two practical controls from the proposal's future-work direction:
- Safety keep: force-retain patches that likely contain safety-critical words.
- Instruction neutralization: mask patches containing jailbreak-style instruction text.

In [ ]:
SAFETY_KEYWORDS = {
    'stop', 'yield', 'warning', 'danger', 'hazard', 'pedestrian', 'school', 'crosswalk',
    'fire', 'exit', 'caution', 'biohazard', 'poison', 'emergency',
}

INSTRUCTION_KEYWORDS = {
    'ignore', 'system prompt', 'developer message', 'jailbreak', 'bypass', 'override',
    'do not follow', 'reveal', 'secret', 'password', 'prompt injection',
}

def text_to_patch_indices(ocr_results, image_shape_hw, grid_size):
    h, w = image_shape_hw
    cell_w = w / grid_size
    cell_h = h / grid_size
    per_text_indices = []

    for item in ocr_results:
        box, text = item[0], str(item[1]).lower() if len(item) > 1 else ''
        xs = [p[0] for p in box]
        ys = [p[1] for p in box]
        x0, x1 = max(min(xs), 0), min(max(xs), w - 1)
        y0, y1 = max(min(ys), 0), min(max(ys), h - 1)

        c0 = max(0, min(int(x0 // cell_w), grid_size - 1))
        c1 = max(0, min(int(x1 // cell_w), grid_size - 1))
        r0 = max(0, min(int(y0 // cell_h), grid_size - 1))
        r1 = max(0, min(int(y1 // cell_h), grid_size - 1))

        idxs = set()
        for r in range(r0, r1 + 1):
            for c in range(c0, c1 + 1):
                idxs.add(r * grid_size + c)
        per_text_indices.append((text, idxs))

    return per_text_indices

def classify_risk_indices(per_text_indices, safety_keywords, instruction_keywords):
    safety_idxs = set()
    instruction_idxs = set()

    for text, idxs in per_text_indices:
        text_norm = normalize_text(text)
        tokens = set(text_norm.split())

        if any(k in text_norm for k in safety_keywords) or len(tokens & safety_keywords) > 0:
            safety_idxs |= idxs
        if any(k in text_norm for k in instruction_keywords) or len(tokens & instruction_keywords) > 0:
            instruction_idxs |= idxs

    return safety_idxs, instruction_idxs

def risk_aware_topk(scores, k, base_topk, safety_force_keep, instruction_blocklist):
    ranked = torch.argsort(scores, descending=True).detach().cpu().tolist()
    final = []

    # Keep safety-critical patches first.
    for idx in sorted(set(int(x) for x in safety_force_keep)):
        if idx not in instruction_blocklist and idx not in final:
            final.append(idx)

    # Keep base selection where not blocked.
    for idx in [int(x) for x in base_topk]:
        if idx not in instruction_blocklist and idx not in final:
            final.append(idx)

    # Fill remaining from score ranking, excluding blocked patches.
    for idx in ranked:
        idx = int(idx)
        if idx in instruction_blocklist:
            continue
        if idx not in final:
            final.append(idx)
        if len(final) >= k:
            break

    return final[:k]

def mask_indices_on_image(image_pil, indices_to_mask, grid_size, fill_value=0):
    arr = np.array(image_pil).copy()
    h, w = arr.shape[:2]
    cell_w = w / grid_size
    cell_h = h / grid_size

    for idx in sorted(set(int(i) for i in indices_to_mask)):
        r, c = divmod(idx, grid_size)
        x0 = int(c * cell_w)
        x1 = int((c + 1) * cell_w)
        y0 = int(r * cell_h)
        y1 = int((r + 1) * cell_h)
        arr[y0:y1, x0:x1] = fill_value
    return Image.fromarray(arr)

print('Risk-awareness helpers loaded.')

In [ ]:
# Single-image risk-aware demo
orig_img_risk = Image.open(image_path).convert('RGB')
img_np_risk = np.array(orig_img_risk)
per_text = text_to_patch_indices(ocr_results, image_shape_hw=img_np_risk.shape[:2], grid_size=grid_size)
safety_idxs, instruction_idxs = classify_risk_indices(
    per_text, SAFETY_KEYWORDS, INSTRUCTION_KEYWORDS
 )

base_idx = [int(x) for x in result.topk_indices.detach().cpu().tolist()]
k_risk = len(base_idx)
risk_idx = risk_aware_topk(
    scores=result.scores,
    k=k_risk,
    base_topk=base_idx,
    safety_force_keep=safety_idxs,
    instruction_blocklist=instruction_idxs,
 )

risk_pruned_img = build_pruned_image(str(image_path), risk_idx, grid_size)
risk_pruned_img = mask_indices_on_image(risk_pruned_img, instruction_idxs, grid_size, fill_value=0)

ans_risk = answer_with_blip(risk_pruned_img, prompt)

risk_summary = {
    'k': k_risk,
    'safety_forced_patch_count': len(safety_idxs),
    'instruction_blocked_patch_count': len(instruction_idxs),
    'risk_topk_count': len(risk_idx),
    'answer_risk_aware_pruned': ans_risk,
}
print(json.dumps(risk_summary, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(orig_img_risk)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(risk_pruned_img)
axes[1].set_title(f'Risk-aware Pruned\nAns: {ans_risk}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: add risk-aware baseline to the existing batch results.
# Run this after the batch benchmark cell if you want risk-aware rows as well.


risk_rows = []
for r in retention_list:
    cfg = PAPITConfig(
        model_id='openai/clip-vit-base-patch32',
        retention_ratio=r,
        anchor_strategy=anchor_strategy,
        device=device,
    )
    pr = PromptAwarePruner(cfg)

    for i, row in samples.iterrows():
        img_path = str(row['image_path'])
        q = str(row['question'])

        out = pr.run(img_path, q)
        base_idx = [int(x) for x in out.topk_indices.detach().cpu().tolist()]
        k = len(base_idx)

        img_np_local = np.array(Image.open(img_path).convert('RGB'))
        vision_cfg_local = pr.model.vision_model.config
        g_local = max(int(getattr(vision_cfg_local, 'image_size', 224) // getattr(vision_cfg_local, 'patch_size', 16)), 1)

        # OCR for this sample, then risk classification.
        _, ocr_res_local = ocr_forced_indices(img_path, grid_size=g_local, lang_list=['en'])
        per_text_local = text_to_patch_indices(ocr_res_local, image_shape_hw=img_np_local.shape[:2], grid_size=g_local)
        safety_local, instr_local = classify_risk_indices(per_text_local, SAFETY_KEYWORDS, INSTRUCTION_KEYWORDS)

        risk_idx_local = risk_aware_topk(
            scores=out.scores,
            k=k,
            base_topk=base_idx,
            safety_force_keep=safety_local,
            instruction_blocklist=instr_local,
        )

        risk_img_local = build_pruned_image(img_path, risk_idx_local, g_local)
        risk_img_local = mask_indices_on_image(risk_img_local, instr_local, g_local, fill_value=0)
        ans_risk_local = answer_with_blip(risk_img_local, q)
        ans_orig_local = answer_with_blip(Image.open(img_path).convert('RGB'), q)

        risk_rows.append({
            'sample_index': int(i),
            'image_path': img_path,
            'question': q,
            'retention_ratio': r,
            'risk_safety_forced_count': len(safety_local),
            'risk_instruction_blocked_count': len(instr_local),
            'answer_orig': ans_orig_local,
            'answer_risk': ans_risk_local,
            'risk_same_as_orig': int(normalize_text(ans_risk_local) == normalize_text(ans_orig_local)),
        })

risk_df = pd.DataFrame(risk_rows)
display(risk_df.head(10))

risk_csv = out_dir / 'benchmark_risk_aware.csv'
risk_df.to_csv(risk_csv, index=False)
print('Saved risk-aware results:', risk_csv.resolve())

## 16) Dataset Bootstrap (Optional: Hugging Face Samples)

Creates a benchmark CSV from small samples of public VQA datasets for quick experiments.

If dataset access fails, keep using your custom benchmark_samples.csv.

In [ ]:
# Optional install for dataset bootstrap.
!pip -q install datasets

In [ ]:
from datasets import load_dataset

# Choose one quick source below: 'textvqa' or 'vqav2'
dataset_source = 'textvqa'
max_rows = 20

rows = []

def try_load_first_available(candidates):
    last_err = None
    for name, split in candidates:
        try:
            ds_local = load_dataset(name, split=split)
            print(f'Loaded dataset: {name} ({split})')
            return ds_local
        except Exception as e:
            print(f'Skipped {name}: {type(e).__name__}: {e}')
            last_err = e
    raise RuntimeError(f'No dataset candidate could be loaded. Last error: {last_err}')

if dataset_source == 'textvqa':
    # Some older dataset script repos are no longer supported by datasets>=4.
    # Try multiple hosted variants first.
    textvqa_candidates = [
        ('lmms-lab/textvqa', 'validation[:50]'),
        ('HuggingFaceM4/TextVQA', 'validation[:50]'),
        ('facebook/textvqa', 'validation[:50]'),
    ]

    try:
        ds = try_load_first_available(textvqa_candidates)
        for ex in ds:
            image_obj = ex.get('image', None)
            q = ex.get('question', '')
            answers = ex.get('answers', [])
            gold = answers[0] if isinstance(answers, list) and len(answers) > 0 else ''

            if image_obj is None or q == '':
                continue

            out_dir = Path('dataset_cache/textvqa_images')
            out_dir.mkdir(parents=True, exist_ok=True)
            img_path = out_dir / f"{len(rows):05d}.png"
            image_obj.save(img_path)

            rows.append({
                'image_path': str(img_path),
                'question': q,
                'answer': str(gold),
            })
            if len(rows) >= max_rows:
                break
    except Exception as e:
        print('TextVQA bootstrap unavailable in this runtime:', e)
        print('Falling back to one-row local sample. You can still run the benchmark pipeline.')
        rows.append({
            'image_path': str(image_path),
            'question': prompt,
            'answer': '',
        })

elif dataset_source == 'vqav2':
    vqav2_candidates = [
        ('HuggingFaceM4/VQAv2', 'validation[:50]'),
        ('Graphcore/vqa', 'validation[:50]'),
    ]
    try:
        ds = try_load_first_available(vqav2_candidates)
        for ex in ds:
            image_obj = ex.get('image', None)
            q = ex.get('question', '')
            answers = ex.get('answers', [])
            gold = answers[0] if isinstance(answers, list) and len(answers) > 0 else ''

            if image_obj is None or q == '':
                continue

            out_dir = Path('dataset_cache/vqav2_images')
            out_dir.mkdir(parents=True, exist_ok=True)
            img_path = out_dir / f"{len(rows):05d}.png"
            image_obj.save(img_path)

            rows.append({
                'image_path': str(img_path),
                'question': q,
                'answer': str(gold),
            })
            if len(rows) >= max_rows:
                break
    except Exception as e:
        print('VQAv2 bootstrap unavailable in this runtime:', e)
        print('Falling back to one-row local sample. You can still run the benchmark pipeline.')
        rows.append({
            'image_path': str(image_path),
            'question': prompt,
            'answer': '',
        })
else:
    raise ValueError("dataset_source must be 'textvqa' or 'vqav2'")

bootstrap_df = pd.DataFrame(rows)
display(bootstrap_df.head())

bootstrap_csv = Path('benchmark_samples_bootstrap.csv')
bootstrap_df.to_csv(bootstrap_csv, index=False)
print('Saved bootstrap CSV:', bootstrap_csv.resolve())
print('Set dataset_csv = benchmark_samples_bootstrap.csv in the batch benchmark cell to use it.')

## 17) Auto-Generate Report Snippet

Generates a markdown summary from benchmark CSV outputs for direct use in your report.

In [ ]:
from datetime import date

summary_path = Path('outputs/benchmark_summary.csv')
risk_path = Path('outputs/benchmark_risk_aware.csv')

if not summary_path.exists():
    raise FileNotFoundError('Run the batch benchmark first so outputs/benchmark_summary.csv exists.')

sum_df = pd.read_csv(summary_path)
best_idx = sum_df['avg_ocr_same_as_orig'].idxmax()
best_row = sum_df.loc[best_idx]

risk_note = 'Risk-aware benchmark not run yet.'
if risk_path.exists():
    risk_df_local = pd.read_csv(risk_path)
    risk_note = f"Risk-aware rows: {len(risk_df_local)}; avg risk_same_as_orig={risk_df_local['risk_same_as_orig'].mean():.3f}"

md_lines = []
md_lines.append('## Implementation Progress (Auto-Generated)')
md_lines.append(f"- Date: {date.today().isoformat()}")
md_lines.append('- Notebook: colab_papit_demo.ipynb')
md_lines.append('- Implemented modules: prompt-aware pruning, OCR-guided retention, random baseline, risk-aware prototype, batch benchmarking')
md_lines.append('')
md_lines.append('### Best Retention (by OCR-guided consistency to original answer)')
md_lines.append(f"- retention_ratio: {best_row['retention_ratio']}")
md_lines.append(f"- avg_ocr_same_as_orig: {best_row['avg_ocr_same_as_orig']:.3f}")
md_lines.append(f"- avg_base_same_as_orig: {best_row['avg_base_same_as_orig']:.3f}")
md_lines.append(f"- avg_random_same_as_orig: {best_row['avg_random_same_as_orig']:.3f}")
md_lines.append(f"- avg_ocr_cov_pct: {best_row['avg_ocr_cov_pct']:.2f}")
md_lines.append(f"- avg_base_cov_pct: {best_row['avg_base_cov_pct']:.2f}")
md_lines.append('')
md_lines.append('### Risk-Aware Status')
md_lines.append(f"- {risk_note}")
md_lines.append('')
md_lines.append('### Remaining Gaps')
md_lines.append('- True token-level LLaVA internal integration remains pending.')
md_lines.append('- Official full-dataset protocol runs (GQA/VQAv2/TextVQA) remain pending.')

report_snippet = '\n'.join(md_lines)
print(report_snippet)

snippet_path = Path('outputs/report_snippet.md')
snippet_path.write_text(report_snippet, encoding='utf-8')
print('Saved report snippet:', snippet_path.resolve())

## 18) Efficiency Benchmark (Latency, Memory, Token Budget)

This section adds the missing compute analysis reviewers expect.

It reports:
- token reduction ratio,
- pruning latency,
- proxy QA latency,
- peak GPU memory (if CUDA is available).

In [ ]:
import time
import numpy as np
import pandas as pd
from pathlib import Path

def measure_one_variant(img_path, question, retention_ratio=0.5, force_ocr=False, risk_aware=False, runs=3):
    cfg = PAPITConfig(
        model_id='openai/clip-vit-base-patch32',
        retention_ratio=retention_ratio,
        anchor_strategy='global_mean',
        device=device,
    )
    pr = PromptAwarePruner(cfg)

    # Warmup pass for more stable timings.
    _ = pr.run(img_path, question)

    prune_times = []
    qa_times = []
    mem_peaks_mb = []

    keep_counts = []
    total_counts = []

    for run_idx in range(runs):
        if torch.cuda.is_available() and device == 'cuda':
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        out = pr.run(img_path, question)

        vision_cfg_local = pr.model.vision_model.config
        grid_local = max(int(getattr(vision_cfg_local, 'image_size', 224) // getattr(vision_cfg_local, 'patch_size', 16)), 1)

        base_idx = [int(x) for x in out.topk_indices.detach().cpu().tolist()]
        final_idx = base_idx

        if force_ocr or risk_aware:
            forced_local, ocr_res_local = ocr_forced_indices(img_path, grid_local, lang_list=['en'])
            final_idx = merge_topk_with_forced(out.scores, out.topk_indices, k=len(base_idx), forced_indices=forced_local)

            if risk_aware:
                img_np_local = np.array(Image.open(img_path).convert('RGB'))
                per_text_local = text_to_patch_indices(ocr_res_local, image_shape_hw=img_np_local.shape[:2], grid_size=grid_local)
                safety_local, instr_local = classify_risk_indices(per_text_local, SAFETY_KEYWORDS, INSTRUCTION_KEYWORDS)
                final_idx = risk_aware_topk(
                    scores=out.scores,
                    k=len(base_idx),
                    base_topk=final_idx,
                    safety_force_keep=safety_local,
                    instruction_blocklist=instr_local,
                )

        t1 = time.perf_counter()
        prune_times.append(t1 - t0)

        pruned_img = build_pruned_image(img_path, final_idx, grid_local)

        tq0 = time.perf_counter()
        _ = answer_with_blip(pruned_img, question)
        tq1 = time.perf_counter()
        qa_times.append(tq1 - tq0)

        if torch.cuda.is_available() and device == 'cuda':
            torch.cuda.synchronize()
            mem_peaks_mb.append(torch.cuda.max_memory_allocated() / (1024 ** 2))
        else:
            mem_peaks_mb.append(np.nan)

        keep_counts.append(len(final_idx))
        total_counts.append(int(out.patch_tokens.shape[0]))

    keep_mean = float(np.mean(keep_counts))
    total_mean = float(np.mean(total_counts))

    return {
        'retention_ratio': retention_ratio,
        'force_ocr': int(force_ocr),
        'risk_aware': int(risk_aware),
        'avg_tokens_kept': keep_mean,
        'avg_total_tokens': total_mean,
        'token_keep_ratio': keep_mean / max(total_mean, 1.0),
        'avg_prune_latency_sec': float(np.mean(prune_times)),
        'avg_qa_latency_sec': float(np.mean(qa_times)),
        'avg_end_to_end_sec': float(np.mean(np.array(prune_times) + np.array(qa_times))),
        'avg_peak_gpu_mem_mb': float(np.nanmean(mem_peaks_mb)) if not np.isnan(np.nanmean(mem_peaks_mb)) else np.nan,
    }

print('Efficiency benchmark helpers loaded.')

In [ ]:
# Run efficiency benchmark on the current image and prompt.
# You can expand retention_grid or runs for stronger evidence.
retention_grid = [1.0, 0.75, 0.5, 0.25]
runs_per_setting = 2

eff_rows = []
for rr in retention_grid:
    eff_rows.append(measure_one_variant(str(image_path), prompt, retention_ratio=rr, force_ocr=False, risk_aware=False, runs=runs_per_setting))
    eff_rows.append(measure_one_variant(str(image_path), prompt, retention_ratio=rr, force_ocr=True, risk_aware=False, runs=runs_per_setting))
    eff_rows.append(measure_one_variant(str(image_path), prompt, retention_ratio=rr, force_ocr=True, risk_aware=True, runs=runs_per_setting))

eff_df = pd.DataFrame(eff_rows).sort_values(['retention_ratio', 'force_ocr', 'risk_aware'], ascending=[False, True, True]).reset_index(drop=True)
display(eff_df)

out_dir = Path('outputs')
out_dir.mkdir(parents=True, exist_ok=True)
eff_csv = out_dir / 'efficiency_benchmark.csv'
eff_df.to_csv(eff_csv, index=False)
print('Saved efficiency benchmark:', eff_csv.resolve())

## 19) Ablation Study (Module Contribution Table)

This section creates a reviewer-style ablation table for:
- base prompt-aware pruning,
- + anchor strategy,
- + OCR keep,
- + safety keep,
- + injection mask.

If a gold answer is available in your CSV, EM/F1 are also reported.

In [ ]:
# Dataset for ablation. Reuse your benchmark CSV for stronger evidence.
ablation_csv = 'benchmark_samples_bootstrap.csv' if Path('benchmark_samples_bootstrap.csv').exists() else 'benchmark_samples.csv'
max_ablation_samples = 20
ablation_retention = 0.25

samples_ab = pd.read_csv(ablation_csv)
if max_ablation_samples is not None:
    samples_ab = samples_ab.head(max_ablation_samples).copy()

def run_ablation_variant(row, use_anchor, use_ocr, use_safety, use_injection_mask):
    img_path = str(row['image_path'])
    q = str(row['question'])
    gold = str(row['answer']) if 'answer' in row and str(row['answer']) != 'nan' else ''

    cfg = PAPITConfig(
        model_id='openai/clip-vit-base-patch32',
        retention_ratio=ablation_retention,
        anchor_strategy='global_mean' if use_anchor else 'none',
        device=device,
    )
    pr = PromptAwarePruner(cfg)
    out = pr.run(img_path, q)

    idx = [int(x) for x in out.topk_indices.detach().cpu().tolist()]

    vision_cfg_local = pr.model.vision_model.config
    grid_local = max(int(getattr(vision_cfg_local, 'image_size', 224) // getattr(vision_cfg_local, 'patch_size', 16)), 1)

    forced_local = []
    ocr_res_local = []
    if use_ocr or use_safety or use_injection_mask:
        forced_local, ocr_res_local = ocr_forced_indices(img_path, grid_local, lang_list=['en'])

    if use_ocr:
        idx = merge_topk_with_forced(out.scores, out.topk_indices, k=len(idx), forced_indices=forced_local)

    blocked_count = 0
    safety_count = 0
    if use_safety or use_injection_mask:
        img_np_local = np.array(Image.open(img_path).convert('RGB'))
        per_text_local = text_to_patch_indices(ocr_res_local, image_shape_hw=img_np_local.shape[:2], grid_size=grid_local)
        safety_local, instr_local = classify_risk_indices(per_text_local, SAFETY_KEYWORDS, INSTRUCTION_KEYWORDS)

        if use_safety:
            idx = risk_aware_topk(
                scores=out.scores,
                k=len(idx),
                base_topk=idx,
                safety_force_keep=safety_local,
                instruction_blocklist=set(),
            )
            safety_count = len(safety_local)

        if use_injection_mask:
            idx = [i for i in idx if i not in instr_local]
            while len(idx) < len(out.topk_indices):
                for cand in torch.argsort(out.scores, descending=True).detach().cpu().tolist():
                    cand = int(cand)
                    if cand not in instr_local and cand not in idx:
                        idx.append(cand)
                    if len(idx) >= len(out.topk_indices):
                        break
            idx = idx[:len(out.topk_indices)]
            blocked_count = len(instr_local)

    pruned_img = build_pruned_image(img_path, idx, grid_local)
    ans = answer_with_blip(pruned_img, q)
    ans_orig_local = answer_with_blip(Image.open(img_path).convert('RGB'), q)

    out_row = {
        'same_as_orig': int(normalize_text(ans) == normalize_text(ans_orig_local)),
        'safety_forced_count': safety_count,
        'injection_blocked_count': blocked_count,
    }

    if len(gold.strip()) > 0:
        out_row['em'] = exact_match(ans, gold)
        out_row['f1'] = token_f1(ans, gold)

    return out_row

variants = [
    ('base_prompt_prune',            False, False, False, False),
    ('plus_anchor',                  True,  False, False, False),
    ('plus_anchor_plus_ocr',         True,  True,  False, False),
    ('plus_anchor_ocr_safety_keep',  True,  True,  True,  False),
    ('full_with_injection_mask',     True,  True,  True,  True),
]

abl_rows = []
for name, a, o, s, m in variants:
    metric_rows = []
    for _, row in samples_ab.iterrows():
        metric_rows.append(run_ablation_variant(row, use_anchor=a, use_ocr=o, use_safety=s, use_injection_mask=m))

    df_local = pd.DataFrame(metric_rows)
    row_out = {
        'variant': name,
        'n_samples': len(df_local),
        'avg_same_as_orig': df_local['same_as_orig'].mean(),
        'avg_safety_forced_count': df_local['safety_forced_count'].mean(),
        'avg_injection_blocked_count': df_local['injection_blocked_count'].mean(),
    }
    if 'em' in df_local.columns:
        row_out['avg_em'] = df_local['em'].mean()
    if 'f1' in df_local.columns:
        row_out['avg_f1'] = df_local['f1'].mean()

    abl_rows.append(row_out)

ablation_df = pd.DataFrame(abl_rows)
display(ablation_df)

ablation_csv_out = Path('outputs/ablation_table.csv')
ablation_df.to_csv(ablation_csv_out, index=False)
print('Saved ablation table:', ablation_csv_out.resolve())